# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [1]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [10]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Fetching .mod files:   1%|          | 40/5133 [00:02<05:44, 14.77it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod - HTTP 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod


Fetching .mod files:   1%|          | 50/5133 [00:02<04:56, 17.13it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod - HTTP 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod
❌ Failed to fetch https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod - HTTP 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod


Fetching .mod files:   2%|▏         | 97/5133 [00:05<04:42, 17.82it/s]

...

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod - HTTP 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod


Fetching .mod files: 100%|█████████▉| 5130/5133 [06:39<00:00, 11.32it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod - HTTP 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod


Fetching .mod files: 100%|██████████| 5133/5133 [06:40<00:00, 12.83it/s]


⚠️ Some downloads failed. Check ../data/failed_downloads.json for details.

✅ All .mod files (including failures) saved in ../data/mod_files.json


# Unzipping the private mod (Error 403) from DropBox

In [43]:
# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

import json
df = pd.read_json("../data/mod_files.json")
df_403 =  df[df["error_code"] == 403].copy()
df_403["mod_model_id"] = df_403["url"].apply(extract_model_id)



In [56]:
#Unable to find zips for the last 3 
zip_names_to_extract.sort()
zip_names_to_extract
#116081
#156034
#185115

[37970,
 38229,
 38235,
 38239,
 38240,
 38241,
 58173,
 65218,
 66276,
 97882,
 116081,
 156034,
 185115]

In [57]:
import os
import zipfile

# Define the folder containing zip files
zip_folder = "../data/dropbox_403"
output_folder = "../data/dropbox_403_unzipped"  # Change this to where you want to extract the files

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Iterate through all files in the folder
for file in os.listdir(zip_folder):
    if file.endswith(".zip"):  # Check if it's a zip file
        zip_path = os.path.join(zip_folder, file)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(output_folder)
        print(f"✅ Extracted: {file}")

print("\n🎉 All zip files have been extracted!")



✅ Extracted: 38240.zip
✅ Extracted: 38239.zip
✅ Extracted: 38229.zip
✅ Extracted: 65218.zip
✅ Extracted: 37970.zip
✅ Extracted: 38241.zip
✅ Extracted: 58173.zip
✅ Extracted: 97882.zip
✅ Extracted: 38235.zip
✅ Extracted: 66276.zip

🎉 All zip files have been extracted!


# Read JSON file

In [5]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option("display.max_columns", None)

In [6]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [7]:
# Function to extract mod directory from the URL
def extract_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all USEION occurrences, stopping before VALENCE
def extract_use_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"USEION\s+([^\s]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    return list(set(matches)) if matches else None  # Remove duplicates


#todo: fix this
# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def extract_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def extract_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def extract_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def extract_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def extract_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def extract_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def extract_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def extract_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  


def has_net_receive(content):
    """
    Checks if a MOD file contains a NET_RECEIVE block.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        int: 1 if NET_RECEIVE exists, otherwise 0.
    """
    if pd.isna(content):  # Handle missing content
        return 0

    # Search for NET_RECEIVE, allowing optional parameters inside parentheses
    match = re.search(r"^\s*NET_RECEIVE\s*\(?.*?\)?\s*\{", content, re.MULTILINE)

    return 1 if match else 0

def extract_point_process(content):
    """
    Extracts the POINT_PROCESS name from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        str or None: The extracted POINT_PROCESS name, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Extract the POINT_PROCESS name, ignoring comments
    match = re.search(r"^\s*POINT_PROCESS\s+([^\n:]+)", content, re.MULTILINE)

    return match.group(1).strip() if match else None


    
# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found

In [8]:
# Load JSON into DataFrame
df = pd.read_json("../data/mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_dir)
df["mod_name"] = df["url"].apply(extract_fname)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_suffix)
df["mod_use_ion"] = df["content"].apply(extract_use_ion)
df["mod_read_ion"] = df["content"].apply(extract_read_ion)
df["mod_write_ion"] = df["content"].apply(extract_write_ion)
df["write_current"] = df["mod_write_ion"].apply(write_current_yn)
df["mod_nonspecific_current"] = df["content"].apply(extract_nonspecific_current)
df["mod_range"] = df["content"].apply(extract_range)
df["mod_global"] = df["content"].apply(extract_global)
df["mod_parameter"] = df["content"].apply(extract_parameter)
df["mod_state"] = df["content"].apply(extract_state)
df["mod_net_receive"] = df["content"].apply(has_net_receive)
df["mod_point_process"] = df["content"].apply(extract_point_process)


In [19]:
df = pd.read_json("../data/mod_files.json")

In [21]:
5133-174

4959

In [59]:
df[df["error_code"]==403]

,row_id,file_hash,raw_sha,count,url,download_url,content,error_code
37,38,123e4283498a834dd9a539b3cf8789fd6aa4c4d4e3f754...,c87f99410a0162b31dcae7e3f49bc69c6ff7640ba4ce70...,4,https://modeldb.science/58173?tab=2&file=b05oc...,https://modeldb.science/getModelFile?model=581...,None,403
46,47,e305d8d09c4304604697436072876d1e6188369e1b672f...,4793e433f250d7ebf26c797ebffabce012561bb19ace8d...,9,https://modeldb.science/65218?tab=2&file=nrntr...,https://modeldb.science/getModelFile?model=652...,None,403
49,50,9bd6920e648b7098c63105ea47b1bb46b7396b59191b15...,3bd115893a13eebba15d55ea4c2407e36664bf465b9bb8...,7,https://modeldb.science/38241?tab=2&file=CA1_m...,https://modeldb.science/getModelFile?model=382...,None,403
141,142,fcecb8a1c0f23ec3496ba53c568493ae07b3ee2d862752...,361d231de1462b51040192c17083fba26d27d50b4f9e6d...,2,https://modeldb.science/58173?tab=2&file=b05oc...,https://modeldb.science/getModelFile?model=581...,None,403
184,185,ae199f5fbb2be25de56e4299e0e57b428007c0b747b772...,150e5241060418e203bef2e25d16be24cfb8c82f0c8a8c...,2,https://modeldb.science/97882?tab=2&file=TOM/k...,https://modeldb.science/getModelFile?model=978...,None,403
...,...,...,...,...,...,...,...,...
4900,4901,d04098891571faf61364f38e5b8c95a2634da09e5d6a9d...,ff62e0dcc7f851fe93dcc80ab164393c87603426725dee...,17,https://modeldb.science/116081?tab=2&file=mod-...,https://modeldb.science/getModelFile?model=116...,None,403
5038,5039,6c585c84e0a0edb4ebe572ef0e9bdd951ba85ccb3a9149...,2a0863b902df66de33be5ce9b7950bb57b7c7b93b9b11a...,1,https://modeldb.science/38229?tab=2&file=kcum.mod,https://modeldb.science/getModelFile?model=382...,None,403
5043,5044,52d195c11e3261a03174ca6aed9916203a28320a104b2d...,327bf47437a68dee390c84ee2f1aaaa6d2c6ff28f17869...,2,https://modeldb.science/38235?tab=2&file=CoHCN...,https://modeldb.science/getModelFile?model=382...,None,403
5121,5122,bdb7dc3e8c2504a91e5d5b222ba72be3eb749caaade255...,13ade1cfcbf5b008326b0c161a11ab4054b366bbf7914f...,3,https://modeldb.science/58173?tab=2&file=b05oc...,https://modeldb.science/getModelFile?model=581...,None,403


In [ ]:
View(df.loc[:,["mod_title","mod_read_ion","mod_write_ion","mod_point_process"]],100)

In [20]:
View(df.loc[99, ["url","mod_read_ion"]])

url             https://modeldb.science/223649?tab=2&file=AlturkiEtAl2016/1_Hemond/Segregated/cal2.mod
mod_read_ion                                                                                [cai, cao]
Name: 99, dtype: object

In [ ]:
View(df.loc[470,["url"]])

In [ ]:
#df[(df["mod_exclude"] == "not_found") | (df["mod_exclude"] == "x86_64")].shape

# Flagging Issues

In [ ]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)
#row_id 99 - need to fix use_ion

# Questions

In [ ]:
#Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? 
#what's the best way to capture 
"""example of a function that captures both active and commented out variables. I think its cleaner to capture active only?
import re
import pandas as pd

# Function to extract RANGE variables based on mode
def extract_range(content, mode="all"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")

"""

In [ ]:
View(df.loc[481,["url","mod_state"]],50)

In [ ]:
import re





In [61]:
! git rm -r --cached ../data/

rm 'data/NaP.mod'
rm 'data/failed_downloads.json'
rm 'data/mod_files.json'
rm 'data/model_db_annotations.xlsx'


In [ ]:
! git add .
! git commit -m "removed dictioanry from read/write current"
! git push 

In [ ]:
import re




In [ ]:
import re




In [ ]:
sample = """ 

NEURON {
	POINT_PROCESS interV2pyrDDANE_STFD
	USEION ca READ eca,ica
	NONSPECIFIC_CURRENT igaba
	RANGE initW
	RANGE Cdur_gaba, AlphaTmax_gaba, Beta_gaba, Erev_gaba, gbar_gaba, W, on_gaba, g_gaba
	RANGE eca, tauCa, Icatotal
	RANGE ICag, P0g, fCag
	RANGE Cainf, pooldiam, z
	RANGE lambda1, lambda2, threshold1, threshold2
	RANGE fmax, fmin, Wmax, Wmin, maxChange, normW, scaleW, srcid, destid
	RANGE pregid,postgid, thr_rp
	RANGE F, f, tauF, D1, d1, tauD1, D2, d2, tauD2
	RANGE facfactor
}

"""

In [ ]:
extract_point_process(sample)